In [ ]:
%pip install -U pandas numpy scipy scikit-learn joblib tabulate openpyxl

In [ ]:
import os
import json
import zipfile
import shutil
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

# For final paper, 1000 is better.
# If the cell is too slow, first test with 100 or 200, then rerun with 1000.
N_PERMUTATIONS_STABILITY = 1000

BASE_DIR = Path("sodium_cathode_leakage_corrected_outputs")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "outputs"

TABLE_DIR = OUTPUT_DIR / "tables"
FIGDATA_DIR = OUTPUT_DIR / "figure_data"
AUDIT_DIR = OUTPUT_DIR / "audit"
TEXT_DIR = OUTPUT_DIR / "manuscript_text"
MODELS_DIR = OUTPUT_DIR / "models"

for folder in [
    BASE_DIR,
    INPUT_DIR,
    OUTPUT_DIR,
    TABLE_DIR,
    FIGDATA_DIR,
    AUDIT_DIR,
    TEXT_DIR,
    MODELS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Notebook 06 output folder:", OUTPUT_DIR.resolve())

In [ ]:
def find_first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(
        "None of these files were found:\n" + "\n".join(str(x) for x in candidates)
    )


OUTPUTS_ZIP = find_first_existing([
    Path("outputs.zip"),
    Path("/mnt/data/outputs.zip"),
])

EXTRACT_DIR = INPUT_DIR / "notebook03_outputs_extracted"

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(OUTPUTS_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

SOURCE_OUTPUTS_DIR = EXTRACT_DIR / "outputs"

if not SOURCE_OUTPUTS_DIR.exists():
    SOURCE_OUTPUTS_DIR = EXTRACT_DIR

print("Loaded:", OUTPUTS_ZIP)
print("Using:", SOURCE_OUTPUTS_DIR)

print("\nFiles detected:")
for p in sorted(SOURCE_OUTPUTS_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(SOURCE_OUTPUTS_DIR))

In [ ]:
def find_inside(root, filename):
    matches = list(Path(root).rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} inside {root}")
    return matches[0]


PATHS = {
    "master": find_inside(SOURCE_OUTPUTS_DIR, "mp_na_electrodes_master_v2_with_ml_predictions_FIXED.csv"),
    "protocol_A_desc": find_inside(SOURCE_OUTPUTS_DIR, "protocol_A_composition_descriptors.csv"),
    "feature_manifest": find_inside(SOURCE_OUTPUTS_DIR, "protocol_feature_manifest.json"),
    "used_features": find_inside(SOURCE_OUTPUTS_DIR, "ml_used_features_by_target_protocol_model.csv"),
    "old_metrics_all": find_inside(SOURCE_OUTPUTS_DIR, "ml_grouped_cv_metrics_all_targets.csv"),
    "old_best": find_inside(SOURCE_OUTPUTS_DIR, "ml_best_model_by_target_protocol.csv"),
    "old_oof": find_inside(SOURCE_OUTPUTS_DIR, "ml_grouped_cv_oof_predictions_all_targets.csv"),
    "old_importance": find_inside(SOURCE_OUTPUTS_DIR, "extratrees_C_strict_feature_importance.csv"),
}

df_master = pd.read_csv(PATHS["master"])
df_comp = pd.read_csv(PATHS["protocol_A_desc"])
df_used_features = pd.read_csv(PATHS["used_features"])
df_old_metrics = pd.read_csv(PATHS["old_metrics_all"])
df_old_best = pd.read_csv(PATHS["old_best"])
df_old_oof = pd.read_csv(PATHS["old_oof"])
df_old_importance = pd.read_csv(PATHS["old_importance"])

with open(PATHS["feature_manifest"], "r", encoding="utf-8") as f:
    feature_manifest = json.load(f)

print("Loaded shapes:")
print("df_master:", df_master.shape)
print("df_comp:", df_comp.shape)
print("df_used_features:", df_used_features.shape)
print("df_old_metrics:", df_old_metrics.shape)
print("df_old_best:", df_old_best.shape)
print("df_old_oof:", df_old_oof.shape)
print("df_old_importance:", df_old_importance.shape)

assert len(df_master) == 416, f"Expected 416 master rows, got {len(df_master)}"
assert df_master["mp_index"].nunique() == 416, "mp_index is not unique."
assert df_master["mp_index"].duplicated().sum() == 0, "Duplicate mp_index detected."
assert len(df_comp) == len(df_master), "Composition descriptor rows do not match master rows."

print("\nProtocol feature counts:")
for k, v in feature_manifest.items():
    if isinstance(v, list):
        print(k, len(v))

In [ ]:
# FIXED Cell 5 — Build unified modelling table safely without duplicate columns

df_comp = df_comp.copy()

# Attach mp_index if missing or overwrite it safely
df_comp["mp_index"] = df_master["mp_index"].values

# IMPORTANT:
# Remove any columns from df_comp that already exist in df_master except mp_index.
# This prevents duplicate columns such as framework_formula, formula_discharge, etc.
overlap_cols = sorted((set(df_comp.columns) & set(df_master.columns)) - {"mp_index"})

if overlap_cols:
    print("Dropping overlapping columns from descriptor table before merge:")
    for c in overlap_cols:
        print(" -", c)
    df_comp = df_comp.drop(columns=overlap_cols)

# Merge composition features into master by mp_index
df_model = df_master.merge(df_comp, on="mp_index", how="left", validate="one_to_one")

# Extra safety: remove duplicate column names if any still exist
duplicated_cols = df_model.columns[df_model.columns.duplicated()].tolist()

if duplicated_cols:
    print("WARNING: duplicate columns found and removed:")
    print(duplicated_cols)
    df_model = df_model.loc[:, ~df_model.columns.duplicated()].copy()

# Rename manifest keys to clean protocol names
PROTOCOL_FEATURES = {
    "A_composition_only": feature_manifest["Protocol_A_composition_only"],
    "B_composition_lattice": feature_manifest["Protocol_B_composition_plus_lattice"],
    "C_strict_post_DFT": feature_manifest["Protocol_C_strict_post_DFT"],
}

TARGETS = {
    "average_voltage": {"unit": "V", "higher_is_better": True},
    "capacity_grav": {"unit": "mAh/g", "higher_is_better": True},
    "energy_grav": {"unit": "Wh/kg", "higher_is_better": True},
    "stability_worst": {"unit": "eV/atom", "higher_is_better": False},
}

GROUP_COL = "framework_formula"

required_cols = ["mp_index", GROUP_COL] + list(TARGETS.keys())

for col in required_cols:
    assert col in df_model.columns, f"Missing required column: {col}"

# Confirm every protocol feature exists
missing_feature_rows = []

for protocol, features in PROTOCOL_FEATURES.items():
    for feat in features:
        if feat not in df_model.columns:
            missing_feature_rows.append({
                "protocol": protocol,
                "missing_feature": feat,
            })

df_missing_features = pd.DataFrame(missing_feature_rows)

if len(df_missing_features) > 0:
    display(df_missing_features)
    raise ValueError("Some protocol features are missing from df_model.")

# Final duplicate-column check
duplicate_check = df_model.columns[df_model.columns.duplicated()].tolist()
assert len(duplicate_check) == 0, f"Duplicate columns still exist: {duplicate_check}"

print("Unified modelling table:", df_model.shape)
print("All protocol features are present.")
print("Grouping column:", GROUP_COL)
print("Unique groups:", df_model[GROUP_COL].astype(str).nunique())

print("\nFirst 10 columns:")
print(df_model.columns[:10].tolist())

In [ ]:
DIRECT_NA_TARGET_FEATURES = {
    "comp__frac_Na",
    "fracA_charge",
    "fracA_discharge",
}

DIRECT_STABILITY_COMPONENTS = {
    "stability_charge",
    "stability_discharge",
}

POST_DFT_FEATURES = {
    "fracA_charge",
    "fracA_discharge",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
}

def feature_family(feature):
    if feature.startswith("comp__"):
        return "composition_descriptor"
    if feature.startswith("host_structure__lattice__"):
        return "lattice_descriptor"
    if feature in POST_DFT_FEATURES:
        return "post_DFT_electrode_descriptor"
    return "other"

def leakage_risk_for_feature(feature):
    risks = []
    decisions = []

    if feature in DIRECT_NA_TARGET_FEATURES:
        risks.append("capacity_grav")
        risks.append("energy_grav")
        decisions.append(
            "EXCLUDE for capacity_grav and energy_grav leakage-corrected analyses because gravimetric capacity is defined by Na stoichiometry; energy is capacity times voltage."
        )

    if feature in DIRECT_STABILITY_COMPONENTS:
        risks.append("stability_worst")
        decisions.append(
            "EXCLUDE for stability_worst leakage-corrected analysis because stability_worst is derived from charged/discharged stability components."
        )
        decisions.append(
            "FLAG for average_voltage as post-DFT cross-target adjacent descriptor; not direct mathematical leakage into voltage."
        )

    if feature == "max_delta_volume":
        decisions.append(
            "KEEP but FLAG as post-DFT electrode descriptor; not a direct mathematical definition of voltage, capacity, energy, or stability_worst."
        )

    if len(decisions) == 0:
        decisions.append("KEEP; no direct target-definition leakage identified.")

    return {
        "risk_for_targets": "; ".join(sorted(set(risks))) if risks else "none_detected",
        "decision": " | ".join(decisions),
    }


audit_rows = []

all_features = sorted(set(sum(PROTOCOL_FEATURES.values(), [])))

for feat in all_features:
    protocols_containing = [
        protocol for protocol, features in PROTOCOL_FEATURES.items()
        if feat in features
    ]

    risk_info = leakage_risk_for_feature(feat)

    audit_rows.append({
        "feature": feat,
        "feature_family": feature_family(feat),
        "protocols_containing_feature": "; ".join(protocols_containing),
        "risk_for_targets": risk_info["risk_for_targets"],
        "decision": risk_info["decision"],
        "is_direct_na_target_feature": feat in DIRECT_NA_TARGET_FEATURES,
        "is_direct_stability_component": feat in DIRECT_STABILITY_COMPONENTS,
        "is_post_DFT_feature": feat in POST_DFT_FEATURES,
    })

df_leakage_audit = pd.DataFrame(audit_rows)

s3_path = TABLE_DIR / "Supplementary_Table_S3_feature_leakage_audit.csv"
df_leakage_audit.to_csv(s3_path, index=False)

print("Saved:", s3_path)
display(df_leakage_audit[df_leakage_audit["risk_for_targets"] != "none_detected"])

In [ ]:
TARGET_SPECIFIC_EXCLUSIONS = {
    "average_voltage": {
        "target_specific_clean": [],
        "stress_no_stability_features": ["stability_charge", "stability_discharge"],
    },
    "capacity_grav": {
        "target_specific_clean": ["comp__frac_Na", "fracA_charge", "fracA_discharge"],
        "strict_no_na_and_stability_features": [
            "comp__frac_Na",
            "fracA_charge",
            "fracA_discharge",
            "stability_charge",
            "stability_discharge",
        ],
    },
    "energy_grav": {
        "target_specific_clean": ["comp__frac_Na", "fracA_charge", "fracA_discharge"],
        "strict_no_na_and_stability_features": [
            "comp__frac_Na",
            "fracA_charge",
            "fracA_discharge",
            "stability_charge",
            "stability_discharge",
        ],
    },
    "stability_worst": {
        "target_specific_clean": ["stability_charge", "stability_discharge"],
    },
}

removal_rows = []

for target, variants in TARGET_SPECIFIC_EXCLUSIONS.items():
    for variant, removed in variants.items():
        removal_rows.append({
            "target": target,
            "variant": variant,
            "removed_features": "; ".join(removed) if removed else "none",
            "reason": (
                "Remove direct or target-adjacent features identified in leakage audit."
                if removed else
                "No direct mathematical leakage feature removed for this target."
            ),
        })

df_exclusion_policy = pd.DataFrame(removal_rows)
policy_path = TABLE_DIR / "target_specific_feature_exclusion_policy.csv"
df_exclusion_policy.to_csv(policy_path, index=False)

print("Saved:", policy_path)
display(df_exclusion_policy)

In [ ]:
# FIXED Cell 8 — ML helper functions with safe 1D group handling

def get_models(random_state=42):
    return {
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=300,
                random_state=random_state,
                n_jobs=-1,
                min_samples_leaf=2,
            )),
        ]),
        "ExtraTrees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(
                n_estimators=400,
                random_state=random_state,
                n_jobs=-1,
                min_samples_leaf=1,
            )),
        ]),
        "GradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(
                n_estimators=300,
                random_state=random_state,
                learning_rate=0.03,
                max_depth=3,
            )),
        ]),
    }


def get_1d_column(df, col):
    """
    Safely return a single 1D Series even if duplicate column names exist.
    """
    x = df[col]

    if isinstance(x, pd.DataFrame):
        print(f"WARNING: column '{col}' appeared multiple times. Using the first copy.")
        x = x.iloc[:, 0]

    return x


def safe_spearman(y_true, y_pred):
    try:
        r, p = spearmanr(y_true, y_pred, nan_policy="omit")
        if np.isnan(r):
            return 0.0
        return float(r)
    except Exception:
        return 0.0


def compute_top20_enrichment(y_true, y_pred, higher_is_better=True):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    n = len(y_true)
    k = max(1, int(np.ceil(0.20 * n)))

    if higher_is_better:
        pred_top_idx = np.argsort(y_pred)[-k:]
        true_top_threshold = np.quantile(y_true, 0.80)
        hit = y_true[pred_top_idx] >= true_top_threshold
        base_rate = np.mean(y_true >= true_top_threshold)
    else:
        pred_top_idx = np.argsort(y_pred)[:k]
        true_top_threshold = np.quantile(y_true, 0.20)
        hit = y_true[pred_top_idx] <= true_top_threshold
        base_rate = np.mean(y_true <= true_top_threshold)

    hit_rate = float(np.mean(hit))

    if base_rate == 0:
        enrichment = np.nan
    else:
        enrichment = float(hit_rate / base_rate)

    return hit_rate, enrichment


def clean_feature_list(features, target, removal_variant):
    features = list(features)

    if removal_variant == "original_full_features":
        removed_requested = []
    else:
        removed_requested = TARGET_SPECIFIC_EXCLUSIONS.get(target, {}).get(removal_variant, [])

    final_features = [f for f in features if f not in removed_requested]

    actually_removed = [f for f in features if f in removed_requested]
    missing_requested = [f for f in removed_requested if f not in features]

    return final_features, actually_removed, missing_requested


def run_grouped_cv_for_model(df, target, features, groups, model_name, model, higher_is_better=True):

    # Remove duplicated feature names
    features = list(dict.fromkeys(features))

    meta_cols = [
        "mp_index",
        "battery_formula",
        "formula_discharge",
        "framework_formula",
        "preliminary_family",
        GROUP_COL,
    ]

    use_cols = features + [target] + meta_cols
    use_cols = [c for c in use_cols if c in df.columns]

    # Remove duplicated requested columns while preserving order
    use_cols = list(dict.fromkeys(use_cols))

    data = df[use_cols].copy()

    # Remove duplicated columns in the sliced data, if any
    if data.columns.duplicated().sum() > 0:
        dup_cols = data.columns[data.columns.duplicated()].tolist()
        print("WARNING: duplicate columns inside model data removed:", dup_cols)
        data = data.loc[:, ~data.columns.duplicated()].copy()

    data[target] = pd.to_numeric(get_1d_column(data, target), errors="coerce")
    data = data.dropna(subset=[target]).reset_index(drop=True)

    X = data[features].apply(pd.to_numeric, errors="coerce")
    y = get_1d_column(data, target).values.astype(float)

    # FIX: force group vector to be 1D
    g = (
        get_1d_column(data, GROUP_COL)
        .astype(str)
        .fillna("missing_group")
        .values
        .ravel()
    )

    n_unique_groups = len(pd.unique(g))
    n_splits_eff = min(N_SPLITS, n_unique_groups)

    if n_splits_eff < 2:
        raise ValueError(f"Not enough groups for GroupKFold for target={target}")

    splitter = GroupKFold(n_splits=n_splits_eff)

    y_pred = np.full(len(y), np.nan)
    y_baseline = np.full(len(y), np.nan)

    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups=g), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        fitted = clone(model)
        fitted.fit(X_train, y_train)

        pred = fitted.predict(X_test)
        y_pred[test_idx] = pred

        baseline_value = float(np.median(y_train))
        y_baseline[test_idx] = baseline_value

        fold_rows.append({
            "fold": fold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_train_groups": len(pd.unique(g[train_idx])),
            "n_test_groups": len(pd.unique(g[test_idx])),
        })

    rmse = float(np.sqrt(mean_squared_error(y, y_pred)))
    mae = float(mean_absolute_error(y, y_pred))
    r2 = float(r2_score(y, y_pred))
    spearman = safe_spearman(y, y_pred)

    baseline_rmse = float(np.sqrt(mean_squared_error(y, y_baseline)))
    baseline_mae = float(mean_absolute_error(y, y_baseline))

    top20_hit_rate, top20_enrichment = compute_top20_enrichment(
        y_true=y,
        y_pred=y_pred,
        higher_is_better=higher_is_better,
    )

    pred_cols = [
        "mp_index",
        "battery_formula",
        "formula_discharge",
        "framework_formula",
        "preliminary_family",
        GROUP_COL,
    ]
    pred_cols = [c for c in pred_cols if c in data.columns]
    pred_cols = list(dict.fromkeys(pred_cols))

    pred_df = data[pred_cols].copy()
    pred_df["target"] = target
    pred_df["model"] = model_name
    pred_df["y_true"] = y
    pred_df["y_pred_oof"] = y_pred
    pred_df["abs_error"] = np.abs(y - y_pred)

    metrics = {
        "target": target,
        "model": model_name,
        "n_samples": len(y),
        "n_features": len(features),
        "n_groups": n_unique_groups,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "spearman": spearman,
        "top20_hit_rate": top20_hit_rate,
        "top20_enrichment": top20_enrichment,
        "baseline_rmse": baseline_rmse,
        "baseline_mae": baseline_mae,
    }

    fold_df = pd.DataFrame(fold_rows)

    return metrics, pred_df, fold_df


print("Fixed helper functions loaded.")

In [ ]:
experiment_rows = []

for target in TARGETS:
    for protocol, base_features in PROTOCOL_FEATURES.items():

        # Always run original full feature set for direct before/after comparison
        variants_to_run = ["original_full_features"]

        # Add target-specific clean variant
        variants_to_run.append("target_specific_clean")

        # Add stress variants where defined
        for extra_variant in TARGET_SPECIFIC_EXCLUSIONS.get(target, {}):
            if extra_variant not in variants_to_run:
                variants_to_run.append(extra_variant)

        for variant in variants_to_run:
            clean_features, actually_removed, missing_requested = clean_feature_list(
                features=base_features,
                target=target,
                removal_variant=variant,
            )

            # Avoid exact duplicate experiments when no features were removed
            if variant != "original_full_features" and len(actually_removed) == 0:
                if target != "average_voltage":
                    continue

            experiment_rows.append({
                "target": target,
                "protocol": protocol,
                "variant": variant,
                "n_features_original": len(base_features),
                "n_features_used": len(clean_features),
                "removed_features": "; ".join(actually_removed) if actually_removed else "none",
                "missing_requested_removed_features": "; ".join(missing_requested) if missing_requested else "none",
                "features": clean_features,
            })

df_experiments = pd.DataFrame(experiment_rows)

experiment_manifest_path = AUDIT_DIR / "notebook06_experiment_manifest.csv"
df_experiments.drop(columns=["features"]).to_csv(experiment_manifest_path, index=False)

print("Saved:", experiment_manifest_path)
display(df_experiments.drop(columns=["features"]))

In [ ]:
all_metrics = []
all_predictions = []
all_fold_rows = []

models = get_models(RANDOM_STATE)

for exp_i, exp in df_experiments.iterrows():
    target = exp["target"]
    protocol = exp["protocol"]
    variant = exp["variant"]
    features = exp["features"]

    print(f"\n=== Running {target} | {protocol} | {variant} | features={len(features)} ===")

    for model_name, model in models.items():
        metrics, pred_df, fold_df = run_grouped_cv_for_model(
            df=df_model,
            target=target,
            features=features,
            groups=df_model[GROUP_COL],
            model_name=model_name,
            model=model,
            higher_is_better=TARGETS[target]["higher_is_better"],
        )

        metrics["protocol"] = protocol
        metrics["variant"] = variant
        metrics["removed_features"] = exp["removed_features"]
        metrics["unit"] = TARGETS[target]["unit"]

        pred_df["protocol"] = protocol
        pred_df["variant"] = variant
        pred_df["removed_features"] = exp["removed_features"]

        fold_df["target"] = target
        fold_df["protocol"] = protocol
        fold_df["variant"] = variant
        fold_df["model"] = model_name

        all_metrics.append(metrics)
        all_predictions.append(pred_df)
        all_fold_rows.append(fold_df)

        print(
            f"{model_name:18s} R2={metrics['r2']:.4f} | "
            f"RMSE={metrics['rmse']:.4f} | Spearman={metrics['spearman']:.4f}"
        )

df_metrics_06 = pd.DataFrame(all_metrics)
df_predictions_06 = pd.concat(all_predictions, ignore_index=True)
df_folds_06 = pd.concat(all_fold_rows, ignore_index=True)

metrics_path = TABLE_DIR / "notebook06_grouped_cv_metrics_all_models_all_variants.csv"
pred_path = FIGDATA_DIR / "notebook06_oof_predictions_all_models_all_variants.csv"
folds_path = AUDIT_DIR / "notebook06_grouped_cv_fold_audit.csv"

df_metrics_06.to_csv(metrics_path, index=False)
df_predictions_06.to_csv(pred_path, index=False)
df_folds_06.to_csv(folds_path, index=False)

print("\nSaved:")
print(metrics_path)
print(pred_path)
print(folds_path)

display(df_metrics_06.head())

In [ ]:
# Select best model per target/protocol/variant by R2.
# For stability, still use R2 for consistency, but interpretation will be cautious.
idx = (
    df_metrics_06
    .sort_values(["target", "protocol", "variant", "r2"], ascending=[True, True, True, False])
    .groupby(["target", "protocol", "variant"], as_index=False)
    .head(1)
    .index
)

df_best_06 = df_metrics_06.loc[idx].copy()
df_best_06 = df_best_06.sort_values(["target", "protocol", "variant"]).reset_index(drop=True)

best_path = TABLE_DIR / "notebook06_best_model_by_target_protocol_variant.csv"
df_best_06.to_csv(best_path, index=False)

# Main supplementary table S4
s4_cols = [
    "target",
    "unit",
    "protocol",
    "variant",
    "model",
    "n_samples",
    "n_features",
    "n_groups",
    "removed_features",
    "rmse",
    "mae",
    "r2",
    "spearman",
    "top20_hit_rate",
    "top20_enrichment",
    "baseline_rmse",
    "baseline_mae",
]

df_s4 = df_best_06[s4_cols].copy()

for col in ["rmse", "mae", "r2", "spearman", "top20_hit_rate", "top20_enrichment", "baseline_rmse", "baseline_mae"]:
    df_s4[col] = pd.to_numeric(df_s4[col], errors="coerce").round(5)

s4_path = TABLE_DIR / "Supplementary_Table_S4_leakage_corrected_ml_performance.csv"
df_s4.to_csv(s4_path, index=False)

print("Saved:", best_path)
print("Saved:", s4_path)

display(df_s4)

In [ ]:
comparison_rows = []

for target in ["capacity_grav", "energy_grav", "stability_worst", "average_voltage"]:
    for protocol in ["A_composition_only", "B_composition_lattice", "C_strict_post_DFT"]:
        full = df_s4[
            (df_s4["target"] == target)
            & (df_s4["protocol"] == protocol)
            & (df_s4["variant"] == "original_full_features")
        ]

        clean = df_s4[
            (df_s4["target"] == target)
            & (df_s4["protocol"] == protocol)
            & (df_s4["variant"] == "target_specific_clean")
        ]

        if full.empty or clean.empty:
            continue

        f = full.iloc[0]
        c = clean.iloc[0]

        comparison_rows.append({
            "target": target,
            "protocol": protocol,
            "full_model": f["model"],
            "clean_model": c["model"],
            "full_n_features": f["n_features"],
            "clean_n_features": c["n_features"],
            "removed_features": c["removed_features"],
            "full_r2": f["r2"],
            "clean_r2": c["r2"],
            "delta_r2_clean_minus_full": c["r2"] - f["r2"],
            "full_spearman": f["spearman"],
            "clean_spearman": c["spearman"],
            "delta_spearman_clean_minus_full": c["spearman"] - f["spearman"],
            "full_rmse": f["rmse"],
            "clean_rmse": c["rmse"],
            "delta_rmse_clean_minus_full": c["rmse"] - f["rmse"],
        })

df_before_after = pd.DataFrame(comparison_rows)

for col in df_before_after.columns:
    if col.startswith("full_") or col.startswith("clean_") or col.startswith("delta_"):
        if col not in ["full_model", "clean_model", "full_n_features", "clean_n_features"]:
            df_before_after[col] = pd.to_numeric(df_before_after[col], errors="coerce").round(5)

before_after_path = TABLE_DIR / "updated_table_2_before_after_leakage_correction.csv"
df_before_after.to_csv(before_after_path, index=False)

print("Saved:", before_after_path)
display(df_before_after)

In [ ]:
importance_rows = []

for _, row in df_best_06.iterrows():
    target = row["target"]
    protocol = row["protocol"]
    variant = row["variant"]
    model_name = row["model"]

    # Only generate importance for best target-specific clean and original C-strict variants
    if variant not in ["target_specific_clean", "original_full_features"]:
        continue

    if model_name not in ["ExtraTrees", "RandomForest", "GradientBoosting"]:
        continue

    exp_row = df_experiments[
        (df_experiments["target"] == target)
        & (df_experiments["protocol"] == protocol)
        & (df_experiments["variant"] == variant)
    ]

    if exp_row.empty:
        continue

    features = exp_row.iloc[0]["features"]

    data = df_model[features + [target]].copy()
    data[target] = pd.to_numeric(data[target], errors="coerce")
    data = data.dropna(subset=[target]).reset_index(drop=True)

    X = data[features].apply(pd.to_numeric, errors="coerce")
    y = data[target].values.astype(float)

    model = clone(models[model_name])
    model.fit(X, y)

    fitted_model = model.named_steps["model"]

    if not hasattr(fitted_model, "feature_importances_"):
        continue

    importances = fitted_model.feature_importances_

    for feat, imp in zip(features, importances):
        leak_row = df_leakage_audit[df_leakage_audit["feature"] == feat]

        if leak_row.empty:
            risk = "not_in_audit"
            decision = ""
        else:
            risk = leak_row.iloc[0]["risk_for_targets"]
            decision = leak_row.iloc[0]["decision"]

        importance_rows.append({
            "target": target,
            "protocol": protocol,
            "variant": variant,
            "model": model_name,
            "feature": feat,
            "importance": float(imp),
            "feature_family": feature_family(feat),
            "risk_for_targets": risk,
            "leakage_audit_decision": decision,
        })

df_importance_06 = pd.DataFrame(importance_rows)

importance_path = TABLE_DIR / "notebook06_feature_importance_with_leakage_flags.csv"
df_importance_06.to_csv(importance_path, index=False)

top20_importance_path = TABLE_DIR / "notebook06_feature_importance_top20_by_target_protocol_variant.csv"

df_top20_imp = (
    df_importance_06
    .sort_values(["target", "protocol", "variant", "importance"], ascending=[True, True, True, False])
    .groupby(["target", "protocol", "variant"])
    .head(20)
    .copy()
)

df_top20_imp.to_csv(top20_importance_path, index=False)

print("Saved:", importance_path)
print("Saved:", top20_importance_path)

display(df_top20_imp.head(60))

In [ ]:
# FIXED Cell 14 — Stability permutation test without sklearn parallel warning spam

import warnings
warnings.filterwarnings(
    "ignore",
    message=".*sklearn.utils.parallel.delayed.*",
    category=UserWarning
)

# Stability permutation test:
# Uses the best target_specific_clean C-strict stability model if available.
# Runs GroupKFold with shuffled labels to estimate null R2 distribution.

stability_best = df_best_06[
    (df_best_06["target"] == "stability_worst")
    & (df_best_06["protocol"] == "C_strict_post_DFT")
    & (df_best_06["variant"] == "target_specific_clean")
].copy()

if stability_best.empty:
    raise ValueError("Could not find best C-strict target_specific_clean model for stability_worst.")

stability_model_name = stability_best.iloc[0]["model"]
stability_observed_r2 = float(stability_best.iloc[0]["r2"])

stability_exp = df_experiments[
    (df_experiments["target"] == "stability_worst")
    & (df_experiments["protocol"] == "C_strict_post_DFT")
    & (df_experiments["variant"] == "target_specific_clean")
].iloc[0]

stability_features = stability_exp["features"]

# IMPORTANT FIX:
# Use n_jobs=1 inside permutation loop.
# This avoids sklearn parallel warning spam and prevents nested parallel overload.
def get_permutation_model(model_name, seed):
    if model_name == "ExtraTrees":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(
                n_estimators=120,
                random_state=seed,
                n_jobs=1,
                min_samples_leaf=1,
            )),
        ])

    if model_name == "RandomForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=120,
                random_state=seed,
                n_jobs=1,
                min_samples_leaf=2,
            )),
        ])

    if model_name == "GradientBoosting":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(
                n_estimators=120,
                random_state=seed,
                learning_rate=0.04,
                max_depth=3,
            )),
        ])

    # Ridge has no internal parallelization issue
    return clone(models[model_name])


stab_data = df_model[stability_features + ["stability_worst", GROUP_COL]].copy()
stab_data["stability_worst"] = pd.to_numeric(stab_data["stability_worst"], errors="coerce")
stab_data = stab_data.dropna(subset=["stability_worst"]).reset_index(drop=True)

X_stab = stab_data[stability_features].apply(pd.to_numeric, errors="coerce")
y_stab = stab_data["stability_worst"].values.astype(float)
g_stab = stab_data[GROUP_COL].astype(str).fillna("missing_group").values.ravel()

splitter = GroupKFold(n_splits=min(N_SPLITS, len(pd.unique(g_stab))))
splits = list(splitter.split(X_stab, y_stab, groups=g_stab))

rng = np.random.default_rng(RANDOM_STATE)
perm_rows = []

print(f"Running stability permutation test with {N_PERMUTATIONS_STABILITY} permutations.")
print("Selected model:", stability_model_name)
print("Observed clean C-strict R2:", stability_observed_r2)

for i in range(N_PERMUTATIONS_STABILITY):
    y_perm = rng.permutation(y_stab)
    y_perm_pred = np.full(len(y_perm), np.nan)

    perm_model = get_permutation_model(stability_model_name, seed=RANDOM_STATE + i)

    for train_idx, test_idx in splits:
        fitted = clone(perm_model)
        fitted.fit(X_stab.iloc[train_idx], y_perm[train_idx])
        y_perm_pred[test_idx] = fitted.predict(X_stab.iloc[test_idx])

    perm_r2 = float(r2_score(y_perm, y_perm_pred))
    perm_spearman = safe_spearman(y_perm, y_perm_pred)

    perm_rows.append({
        "permutation": i + 1,
        "permuted_r2": perm_r2,
        "permuted_spearman": perm_spearman,
    })

    if (i + 1) % 100 == 0:
        print(f"Completed {i + 1}/{N_PERMUTATIONS_STABILITY}")

df_perm = pd.DataFrame(perm_rows)

# One-sided empirical p-value:
# How often shuffled-label R2 is equal to or better than observed real-label R2.
p_value_r2 = (1 + np.sum(df_perm["permuted_r2"] >= stability_observed_r2)) / (len(df_perm) + 1)

perm_summary = pd.DataFrame([{
    "target": "stability_worst",
    "protocol": "C_strict_post_DFT",
    "variant": "target_specific_clean",
    "model": stability_model_name,
    "observed_r2": stability_observed_r2,
    "n_permutations": N_PERMUTATIONS_STABILITY,
    "null_r2_mean": float(df_perm["permuted_r2"].mean()),
    "null_r2_std": float(df_perm["permuted_r2"].std()),
    "null_r2_95th_percentile": float(df_perm["permuted_r2"].quantile(0.95)),
    "empirical_p_value_r2": float(p_value_r2),
}])

perm_data_path = FIGDATA_DIR / "Supplementary_Figure_S1_stability_permutation_null_data.csv"
perm_summary_path = TABLE_DIR / "Supplementary_Table_stability_permutation_test_summary.csv"

df_perm.to_csv(perm_data_path, index=False)
perm_summary.to_csv(perm_summary_path, index=False)

print("Saved:", perm_data_path)
print("Saved:", perm_summary_path)

display(perm_summary)

In [ ]:
def extract_clean_metric(target, protocol="C_strict_post_DFT", variant="target_specific_clean", metric="r2"):
    row = df_s4[
        (df_s4["target"] == target)
        & (df_s4["protocol"] == protocol)
        & (df_s4["variant"] == variant)
    ]
    if row.empty:
        return np.nan
    return float(row.iloc[0][metric])


def extract_full_metric(target, protocol="C_strict_post_DFT", metric="r2"):
    row = df_s4[
        (df_s4["target"] == target)
        & (df_s4["protocol"] == protocol)
        & (df_s4["variant"] == "original_full_features")
    ]
    if row.empty:
        return np.nan
    return float(row.iloc[0][metric])


cap_full_r2 = extract_full_metric("capacity_grav")
cap_clean_r2 = extract_clean_metric("capacity_grav")

energy_full_r2 = extract_full_metric("energy_grav")
energy_clean_r2 = extract_clean_metric("energy_grav")

volt_clean_r2 = extract_clean_metric("average_voltage")
stab_clean_r2 = extract_clean_metric("stability_worst")

cap_full_sp = extract_full_metric("capacity_grav", metric="spearman")
cap_clean_sp = extract_clean_metric("capacity_grav", metric="spearman")

energy_full_sp = extract_full_metric("energy_grav", metric="spearman")
energy_clean_sp = extract_clean_metric("energy_grav", metric="spearman")

stab_p = float(perm_summary.iloc[0]["empirical_p_value_r2"])

ml_text = f"""
# Updated leakage-aware ML interpretation for manuscript

The original C-strict model should not be presented as a clean pre-DFT predictor for capacity or energy. 
The feature-leakage audit identifies `comp__frac_Na`, `fracA_charge`, and `fracA_discharge` as near-target Na-stoichiometry descriptors for `capacity_grav`; because `energy_grav` is derived from capacity and voltage, the same features are also target-adjacent for energy.

In the full-feature C-strict setting, the capacity model gave R2 = {cap_full_r2:.3f} and Spearman = {cap_full_sp:.3f}. 
After removing Na-stoichiometry near-target features, the leakage-corrected C-strict capacity result was R2 = {cap_clean_r2:.3f} and Spearman = {cap_clean_sp:.3f}. 
The full-feature C-strict energy model gave R2 = {energy_full_r2:.3f} and Spearman = {energy_full_sp:.3f}; after the same leakage correction, energy gave R2 = {energy_clean_r2:.3f} and Spearman = {energy_clean_sp:.3f}.

Therefore, the manuscript should frame the full C-strict capacity/energy models as post-DFT reconstruction or decision-support models, not as fair pre-DFT discovery predictors. The leakage-corrected metrics should be reported as the reviewer-safe estimate of predictive performance after removing Na-stoichiometry near-target variables.

For `average_voltage`, the target-specific clean C-strict model gave R2 = {volt_clean_r2:.3f}. For `stability_worst`, the target-specific clean C-strict model gave R2 = {stab_clean_r2:.3f}. The stability model should remain an auxiliary cautionary result. The permutation test for `stability_worst` returned empirical p-value = {stab_p:.4f}; this should be interpreted as evidence that the signal is above shuffled-label null only if p < 0.05, but not as evidence that stability prediction is strong enough for candidate ranking.

Recommended wording:

“Because gravimetric capacity is directly related to Na stoichiometry, we report C-strict capacity and energy performance both with and without Na-content near-target features. The full-feature C-strict models are interpreted as post-DFT decision-support reconstruction models, whereas the leakage-corrected results provide a stricter estimate of predictive performance. Stability prediction remains substantially weaker than voltage, capacity, and energy prediction and is therefore used only as an auxiliary cautionary indicator.”
"""

ml_text_path = TEXT_DIR / "updated_manuscript_ml_interpretation.md"
ml_text_path.write_text(ml_text, encoding="utf-8")

print("Saved:", ml_text_path)
print(ml_text)

In [ ]:
required_outputs = [
    TABLE_DIR / "Supplementary_Table_S3_feature_leakage_audit.csv",
    TABLE_DIR / "Supplementary_Table_S4_leakage_corrected_ml_performance.csv",
    TABLE_DIR / "updated_table_2_before_after_leakage_correction.csv",
    TABLE_DIR / "notebook06_best_model_by_target_protocol_variant.csv",
    TABLE_DIR / "notebook06_grouped_cv_metrics_all_models_all_variants.csv",
    TABLE_DIR / "notebook06_feature_importance_with_leakage_flags.csv",
    TABLE_DIR / "notebook06_feature_importance_top20_by_target_protocol_variant.csv",
    TABLE_DIR / "Supplementary_Table_stability_permutation_test_summary.csv",
    FIGDATA_DIR / "notebook06_oof_predictions_all_models_all_variants.csv",
    FIGDATA_DIR / "Supplementary_Figure_S1_stability_permutation_null_data.csv",
    AUDIT_DIR / "notebook06_experiment_manifest.csv",
    AUDIT_DIR / "notebook06_grouped_cv_fold_audit.csv",
    TEXT_DIR / "updated_manuscript_ml_interpretation.md",
]

audit_rows = []

for p in required_outputs:
    p = Path(p)
    audit_rows.append({
        "file": str(p.relative_to(OUTPUT_DIR)),
        "exists": p.exists(),
        "size_kb": round(p.stat().st_size / 1024, 2) if p.exists() else 0,
    })

df_final_audit_06 = pd.DataFrame(audit_rows)

audit06_path = AUDIT_DIR / "notebook06_final_output_audit.csv"
df_final_audit_06.to_csv(audit06_path, index=False)

display(df_final_audit_06)

missing = df_final_audit_06[df_final_audit_06["exists"] == False]

if len(missing) == 0:
    print("Notebook 06 completed successfully.")
else:
    print("Missing files:")
    display(missing)
    raise FileNotFoundError("Some required Notebook 06 outputs are missing.")

In [ ]:
FINAL_ZIP = Path("sodium_cathode_leakage_corrected_outputs.zip")

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(BASE_DIR))

print("Created:", FINAL_ZIP.resolve())
print("Size MB:", round(FINAL_ZIP.stat().st_size / (1024 * 1024), 3))